# E07 Abalone：混合编码、目标变换与残差

> Issue #9 / Epic #6 / 路线图 E07

## 你将学到
1. `Sex` 类别编码方式影响很大
2. 目标 `Rings` 可做 log1p 再还原
3. 残差分析看是否系统偏差

## 数据
- 本地 `data/abalone.csv` 或 OpenML `abalone`


In [ ]:
QUICK = True
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

from wanglaoshi import VERSIONS as V
from wanglaoshi import JupyterEnv as JE

JE.no_warning()
V.show_env_info()


## 1. 加载 Abalone


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/abalone.csv')
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml('abalone', version=1, as_frame=True, parser='auto')
    df = raw.frame.copy()
    print('OpenML', df.shape)

# 统一列名
rename = {c: c.strip() for c in df.columns}
df = df.rename(columns=rename)
if 'Rings' not in df.columns:
    # OpenML 有时叫 class
    for c in df.columns:
        if c.lower() in ('rings', 'class'):
            df = df.rename(columns={c: 'Rings'})
            break
df['Rings'] = pd.to_numeric(df['Rings'], errors='coerce')
print(df.head())
print(df['Sex'].value_counts() if 'Sex' in df.columns else df.columns)


## 2. EDA


In [ ]:
from wanglaoshi import DataAnalyzer

analyzer = DataAnalyzer(df.dropna())
display(analyzer.basic_statistics())
display(analyzer.missing_value_analysis())


## 3. 编码 Sex + 划分


In [ ]:
from sklearn.model_selection import train_test_split

work = df.dropna().copy()
# one-hot Sex
work = pd.get_dummies(work, columns=['Sex'], drop_first=True)
y = work['Rings']
X = work.drop(columns=['Rings']).apply(pd.to_numeric, errors='coerce')
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
print(Xtr.shape)


## 4. 原尺度 vs log1p 目标


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline = CT.QuickBaseline()

# A) 原尺度
res_a = baseline.run_all_models(Xtr, ytr, Xte, yte, task_type='regression')
cmp_a = baseline.compare_models(res_a, top_n=4)
display(cmp_a)
best_a = cmp_a.iloc[0]['Model']
pred_a = res_a[best_a]['predictions']
print('raw MAE', mean_absolute_error(yte, pred_a))

# B) log1p
ytr_l, yte_l = np.log1p(ytr), np.log1p(yte)
res_b = baseline.run_all_models(Xtr, ytr_l, Xte, yte_l, task_type='regression')
cmp_b = baseline.compare_models(res_b, top_n=4)
display(cmp_b)
best_b = cmp_b.iloc[0]['Model']
pred_b = np.expm1(res_b[best_b]['predictions'])
print('log1p->expm1 MAE', mean_absolute_error(yte, pred_b))


## 5. 残差分析


In [ ]:
import matplotlib.pyplot as plt

resid = yte.to_numpy() - np.asarray(pred_b)
plt.figure(figsize=(6, 4))
plt.scatter(yte, resid, s=10, alpha=0.5)
plt.axhline(0, color='k', lw=1)
plt.xlabel('true Rings'); plt.ylabel('residual (true-pred)')
plt.title('Residual vs Rings')
plt.show()
print('residual mean', resid.mean(), 'std', resid.std())


## 6. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/07_abalone'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred_b)), 'Rings': yte.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_b, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 7. 检查清单
- [x] Sex one-hot
- [x] log1p 目标对比
- [x] 残差图
- [ ] 目标编码 Sex 的折内对比
- [ ] 分年龄段校准
